# STC Jawwy

In [1]:
"""
Here we install libraries that are not installed by default
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [2]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset consists of details about each customer and the movies and/or tv shows watched in addition to the genre.

You are required to work on task three to build a recommendation engine for our platform to Recommend movies to usesrs that they might be interested in¶


In [6]:
from google.colab import files
import pandas as pd
import os

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

filename = list(uploaded.keys())[0]

# Confirm what was actually uploaded
print("Uploaded filename:", filename)

# Just print the sheet names — don't try to read yet
xl = pd.ExcelFile(filename)
print("Sheet names in this file:")
print(xl.sheet_names)

Saving stc TV Data Set_T3.xlsx to stc TV Data Set_T3.xlsx
Uploaded filename: stc TV Data Set_T3.xlsx
Sheet names in this file:
['Sheet1']


In [8]:
import zipfile

print("File size:", os.path.getsize(filename), "bytes")

try:
    with zipfile.ZipFile(filename) as z:
        names = z.namelist()
        print(f"Valid zip container with {len(names)} internal parts")
        for n in names:
            print(" ", n)
except zipfile.BadZipFile:
    print("NOT a valid zip-based file at all — likely corrupted or an old-style .xls")

File size: 36686207 bytes
Valid zip container with 10 internal parts
  [Content_Types].xml
  _rels/.rels
  xl/_rels/workbook.xml.rels
  xl/worksheets/sheet1.xml
  xl/workbook.xml
  xl/sharedStrings.xml
  xl/styles.xml
  xl/theme/theme1.xml
  docProps/core.xml
  docProps/app.xml


In [9]:
dataframe = pd.read_excel(filename, index_col=0, engine='openpyxl')
df = dataframe.copy()
df.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


In [10]:
# check the data shape
dataframe.shape

(1048575, 5)

In [11]:
# display the first 5 rows
dataframe.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


In [12]:
# describe the numeric values in the dataset
dataframe.describe()

,user_id_maped,rating,date_
count,1.048575e+06,1.048575e+06,1048575
mean,1.709266e+04,2.497283e+00,2017-10-04 00:23:20.346183936
min,1.000000e+00,1.000000e+00,2017-03-14 00:00:00
25%,8.253000e+03,1.000000e+00,2017-06-10 00:00:00
50%,1.714900e+04,2.000000e+00,2017-10-14 00:00:00
75%,2.566500e+04,3.000000e+00,2018-01-21 00:00:00
max,3.428000e+04,4.000000e+00,2018-04-30 00:00:00
std,1.003513e+04,1.119837e+00,NaN


In [13]:
# check if any column has null value in the dataset
dataframe.isnull().any()

,0
user_id_maped,False
program_name,False
rating,False
date_,False
program_genre,False


In [14]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [15]:
"""
TODO build your Recommender system to Highlight Programs that usesrs might be interested in
"""

'\nTODO build your Recommender system to Highlight Programs that usesrs might be interested in\n'

## Building the Recommender System

**Approach: item-based collaborative filtering.**

Each row in the dataset is one *viewing event* (a user watching a program on a given date, with a 1-4 `rating`). The same user often watches the same program more than once, so the first step is to collapse this down to one row per `(user, program)` pair.

For every `(user, program)` pair we combine two signals into a single **interaction weight**:
- `avg_rating` — how much the user seemed to like it (mean of their `rating` values for that program)
- `watch_count` — how often they came back to it (repeat viewing is a strong positive signal)

`watch_count` is extremely skewed (a handful of users have 1000+ views of one program, most likely a background/kids channel left running), so it is dampened with `log1p` before being combined with the rating, instead of used raw.

From there:
1. Build a sparse **user × program** matrix of interaction weights.
2. Compute **cosine similarity between programs** based on how similarly they were watched (which users watched them, and how much) — this is the classic "people who watched X also watched Y" signal.
3. Use those similarities to build two things:
   - `similar_programs(name)` — programs most similar to a given program (audience-level, not tied to one user)
   - `recommend_for_user(user_id)` — a personalized top-N for one specific user, based on everything *they* watched

Everything is done with sparse matrices, so it comfortably handles ~11.6K users × ~8K programs / ~440K interactions.

In [18]:
# ============================================================
# SETUP - run this cell once before the cells below.
# It builds the whole recommender: cleans program names, aggregates
# repeat views, builds the sparse user-item matrix, and defines
# find_program() / similar_programs() / recommend_for_user().
#
# If you ever see "NameError: similar_programs/users_arr/... is not
# defined", it means this cell (or the data-loading cells above it)
# hasn't been (re-)run yet in this session - e.g. after a Colab
# runtime restart, all cells need to be re-run top to bottom
# (Runtime > Run all), not just the cell you're currently looking at.
# ============================================================

from scipy import sparse
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# --- Clean up program names ---
# A handful of titles show up with inconsistent casing / spacing
# (e.g. "How to Train Your Dragon" vs "How To Train Your Dragon").
# We fold each group of variants down to its single most common spelling
# so they aren't treated as different programs.
_clean_key = df['program_name'].str.strip().str.lower()
_canonical = df['program_name'].groupby(_clean_key).agg(lambda s: s.value_counts().idxmax())
df['program_name'] = _clean_key.map(_canonical)
print(f"Programs after cleaning: {df['program_name'].nunique()} unique titles")

# --- Aggregate repeat views into one row per (user, program) ---
interactions = (
    df.groupby(['user_id_maped', 'program_name'])
      .agg(avg_rating=('rating', 'mean'), watch_count=('rating', 'size'))
      .reset_index()
)
# Interaction weight = how much they liked it, boosted by how often they rewatched it.
# log1p() dampens the effect of extreme rewatch counts (outliers in watch_count).
interactions['weight'] = interactions['avg_rating'] * (1 + np.log1p(interactions['watch_count']))
print(f"{len(interactions):,} unique (user, program) interactions")

# --- Build the sparse user x program interaction matrix ---
user_cat = pd.Categorical(interactions['user_id_maped'])
item_cat = pd.Categorical(interactions['program_name'])
n_users, n_items = len(user_cat.categories), len(item_cat.categories)

user_item = sparse.csr_matrix(
    (interactions['weight'], (user_cat.codes, item_cat.codes)),
    shape=(n_users, n_items)
)

users_arr = np.array(user_cat.categories)     # position -> user_id
items_arr = np.array(item_cat.categories)     # position -> program_name
user_pos  = {u: i for i, u in enumerate(users_arr)}
item_pos  = {p: i for i, p in enumerate(items_arr)}

# Program vectors (for computing program-to-program similarity), L2-normalized
# so a plain dot product between two rows equals their cosine similarity.
item_user = user_item.T.tocsr()
item_user_l2 = normalize(item_user, norm='l2', axis=1)

print(f"user-item matrix: {user_item.shape[0]:,} users x {user_item.shape[1]:,} programs, "
      f"{user_item.nnz:,} non-zero interactions ({user_item.nnz / (n_users*n_items):.3%} filled)")

# --- Recommender functions ---

def find_program(query, limit=10):
    """Case-insensitive substring search over program titles - handy for
    checking the exact spelling of a title before querying the recommender."""
    q = query.strip().lower()
    return [p for p in items_arr if q in p.lower()][:limit]


def similar_programs(program_name, top_n=5):
    """Programs most similar to `program_name`, based on overlap in who
    watched them and how much ('people who watched X also watched...')."""
    if program_name not in item_pos:
        raise ValueError(f"'{program_name}' not found. Did you mean: {find_program(program_name, 5)}")
    idx = item_pos[program_name]
    sims = cosine_similarity(item_user_l2[idx], item_user_l2).ravel()
    sims[idx] = -1  # exclude the program itself
    top_idx = np.argsort(-sims)[:top_n]
    return pd.DataFrame({'program_name': items_arr[top_idx], 'similarity': sims[top_idx]})


def recommend_for_user(user_id, top_n=5, exclude_watched=True):
    """Personalized top-N recommendations for one user: score every program
    by its similarity to everything that user has already watched, weighted
    by how much they liked/rewatched each one."""
    if user_id not in user_pos:
        raise ValueError(f"user_id {user_id} not found in dataset.")
    u = user_pos[user_id]
    user_weights = user_item[u]
    # (user_weights @ item_user_l2) @ item_user_l2.T is algebraically the same as
    # user_weights @ (full item-item similarity matrix), but never materializes
    # that 8000x8000 matrix - much cheaper.
    scores = user_weights.dot(item_user_l2).dot(item_user_l2.T).toarray().ravel()
    if exclude_watched:
        scores[user_weights.indices] = -np.inf
    top_idx = np.argsort(-scores)[:top_n]
    return pd.DataFrame({'program_name': items_arr[top_idx], 'score': scores[top_idx]})


Programs after cleaning: 8004 unique titles
440,011 unique (user, program) interactions
user-item matrix: 11,578 users x 8,004 programs, 440,011 non-zero interactions (0.475% filled)


In [19]:
# --- Quick sanity check: does the system make sense for an arbitrary program/user? ---
print("Programs similar to 'Trolls':")
display(similar_programs('Trolls', 5))

sample_user = users_arr[100]
print(f"\nPersonalized recommendations for user {sample_user}:")
display(recommend_for_user(sample_user, 5))

Programs similar to 'Trolls':


,program_name,similarity
0,The Mermaid Princess,0.640200
1,Moana,0.621666
2,Surf's Up : WaveMania,0.577931
3,The Jetsons & WWE: Robo-WrestleMania!,0.562403
4,The Boss Baby,0.558401



Personalized recommendations for user 289:


,program_name,score
0,The Walking Dead labtown,143.904481
1,The Walking Dead elf Help,143.406581
2,The Walking Dead Consumed,142.070045
3,The Walking Dead Clear,141.933559
4,The Walking Dead Made To uffer,140.789554


In [16]:
"""
TODO show the recommendations (top 5) for the people who watched "Moana" movie
"""

'\nTODO show the recommendations (top 5) for the people who watched "Moana" movie\n'

In [20]:
# Top 5 programs most similar to "Moana", i.e. what the audience that
# watched Moana also tends to watch:
moana_recs = similar_programs('Moana', top_n=5)
print('Top 5 recommendations for people who watched "Moana":')
moana_recs

Top 5 recommendations for people who watched "Moana":


,program_name,similarity
0,Trolls,0.621666
1,Surf's Up : WaveMania,0.558446
2,The Mermaid Princess,0.520555
3,Storks,0.473771
4,The Boss Baby,0.472536


In [21]:
# Visualize the top 5
fig = px.bar(
    moana_recs.sort_values('similarity'),
    x='similarity', y='program_name', orientation='h',
    title='Top 5 recommendations for people who watched "Moana"',
    labels={'similarity': 'Similarity to Moana', 'program_name': 'Program'}
)
fig.show()

In [22]:
# Find everyone who watched Moana, and show personalized top-5s for a few of them
moana_idx = item_pos['Moana']
moana_watcher_positions = user_item[:, moana_idx].nonzero()[0]
moana_watchers = users_arr[moana_watcher_positions]
print(f"{len(moana_watchers):,} users watched Moana")

rng = np.random.default_rng(42)
sample_watchers = rng.choice(moana_watchers, size=3, replace=False)

for uid in sample_watchers:
    print(f"\nUser {uid}:")
    display(recommend_for_user(uid, top_n=5))

2,173 users watched Moana

User 23347:


,program_name,score
0,Trolls,2.105145
1,Surf's Up : WaveMania,1.891062
2,The Mermaid Princess,1.762754
3,Storks,1.604326
4,The Boss Baby,1.600145



User 3143:


,program_name,score
0,Littlest Pet hop Guilt Tripping,188.568787
1,Littlest Pet hop nipmates,185.845468
2,Littlest Pet hop Un-Vetted,184.397168
3,Littlest Pet hop Two Peas In A Podcast,181.507604
4,Littlest Pet hop In The Loop,180.712405



User 27228:


,program_name,score
0,Shimmer And hine What A Pig Mess,492.880078
1,Shimmer And hine Lights! Camera! Genies!,487.600072
2,Shimmer And hine Ahoy Genies!,487.347776
3,SpongeBob quarePants Tentacle-Vision / I Hear...,487.119622
4,Shimmer And hine Backyard Ballet,486.680379
